# HC-MARL Baseline Training (6 methods x 5 seeds)

**File:** #69 from HC-MARL battle plan

Train all 6 baseline methods (MAPPO, IPPO, MAPPO-Lag, PPO-Lag, CPO, MACPO) for 5M steps each across 5 seeds.

**Requirements:** Colab Pro with T4 GPU (or Kaggle T4 P100)

**Estimated time:** 3-4 hours per seed on T4

## Step 1: Install

In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install gymnasium pettingzoo scipy cvxpy numpy pandas matplotlib pyyaml wandb nbformat jupyter
!pip install omnisafe safety-gymnasium 2>/dev/null || echo "OmniSafe/Safety-Gymnasium optional"


## Step 2: Clone

In [ ]:
# Clone repository (change URL to your repo)
import os

REPO_URL = "https://github.com/ADITYA-WORK-MAITI/hcmarl-project.git"  # <-- UPDATE THIS
PROJECT_DIR = "/content/hcmarl_project"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . 2>/dev/null || echo "Package install skipped"

# Verify
!python -c "import hcmarl; print('HC-MARL package OK')"
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

## Step 3: Drive

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

# Create output directory on Drive
DRIVE_DIR = "/content/drive/MyDrive/hcmarl_results"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_DIR}")


## Step 4: W&B

In [ ]:
# Weights & Biases login (optional but recommended)
import wandb
try:
    wandb.login()
    print("W&B logged in successfully")
except Exception as e:
    print(f"W&B login skipped: {e}")
    print("Training will still work with CSV logging")


## GPU Check

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), 'No GPU detected! Enable GPU in Runtime > Change runtime type'
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu} ({mem:.1f} GB)')


## Training

In [ ]:
# Run all training jobs
!python scripts/run_baselines.py --device cuda


### Or run individual seeds

In [ ]:
# Run a single baseline method + seed
# Uncomment the one you want:
# !python scripts/train.py --config config/mappo_config.yaml --method mappo --seed 0 --device cuda
# !python scripts/train.py --config config/ippo_config.yaml --method ippo --seed 0 --device cuda
# !python scripts/train.py --config config/mappo_lag_config.yaml --method mappo_lag --seed 0 --device cuda
# !python scripts/train.py --config config/ppo_lag_config.yaml --method ppo_lag --seed 0 --device cuda
# !python scripts/train.py --config config/cpo_config.yaml --method cpo --seed 0 --device cuda
# !python scripts/train.py --config config/macpo_config.yaml --method macpo --seed 0 --device cuda


## Evaluation

In [ ]:
# Evaluate trained models
import glob, json

checkpoints = glob.glob('checkpoints/*/seed_*/checkpoint_final.pt')
print(f'Found {len(checkpoints)} checkpoints')

for ckpt in sorted(checkpoints):
    parts = ckpt.replace('\\', '/').split('/')
    method = parts[1]
    seed = parts[2].split('_')[1]
    print(f'\nEvaluating {method} seed={seed}...')
    !python scripts/evaluate.py --checkpoint {ckpt} --config config/hcmarl_full_config.yaml --method {method} --seed {seed} --n-episodes 100


## Save Results to Drive

In [ ]:
# Copy all results to Google Drive
import shutil
for folder in ['checkpoints', 'logs', 'results']:
    src = os.path.join(PROJECT_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied {folder}/ to Drive')
print('\nAll results saved to Google Drive!')
